# 🌌 Galaxy Sizes Across Cosmic Time
### EPS Research High-School Exploration Track — Ages 15-18

The EPS IntZ corpus contains **1,292 galaxies** spanning intermediate redshift.
For **706 KMOS3D galaxies**, the canonical nested record includes a measured
H-band effective radius, `Reff_H_kpc`.

This notebook plots those 706 measured galaxy sizes against spectroscopic
redshift. Galaxies without a usable effective-radius measurement are not
silently treated as zero-size objects.

**What you will learn:**
- How to load a nested astronomical JSON corpus
- How to select records with an actual measurement
- How effective radius varies across redshift
- How to summarize a large sample with binned medians


In [ ]:
# ── Colab setup: canonical FAIR² IntZ corpus ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import urllib.request
    INTZ_FILE = 'intz_corpus_v1b.json'
    INTZ_URL = 'https://zenodo.org/records/21841382/files/intz_corpus_v1b.json'
    if not os.path.exists(INTZ_FILE):
        print(f"Downloading {INTZ_FILE}...")
        urllib.request.urlretrieve(INTZ_URL, INTZ_FILE)
        print(f"  ✓ {INTZ_FILE}")
    INTZ_PATH = INTZ_FILE
else:
    INTZ_PATH = '../intz/intz_corpus_v1b.json'
    print("Running locally — using canonical IntZ v1b corpus.")


In [ ]:
import matplotlib
matplotlib.use('Agg')
import json
import numpy as np
import matplotlib.pyplot as plt

with open(INTZ_PATH) as f:
    corpus = json.load(f)

galaxies = corpus['galaxies']
assert len(galaxies) == 1292

rows = []
for g in galaxies:
    try:
        ident = g['identifiers']
        redshift = g['redshift']
        morph = g['morphology']

        z = redshift['z_spec']
        reff = morph['Reff_H_kpc']
        survey = ident['survey']
        galaxy_id = ident['id']

        if z is None or reff is None:
            continue

        z = float(z)
        reff = float(reff)

        if reff <= 0:
            continue

        rows.append({
            'id': galaxy_id,
            'survey': survey,
            'z': z,
            'Reff': reff
        })

    except (KeyError, TypeError, ValueError):
        continue

assert len(rows) == 706
assert {r['survey'] for r in rows} == {'KMOS3D'}

print(f'IntZ corpus total: {len(galaxies)} galaxies')
print(f'Galaxies with measured Reff_H_kpc: {len(rows)}')
print(f'Surveys in plotted sample: {sorted({r["survey"] for r in rows})}')

zs = np.array([r['z'] for r in rows])
rs = np.array([r['Reff'] for r in rows])

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(zs, rs, s=8, alpha=0.4)

z_bins = np.arange(0.3, 2.8, 0.3)
z_mid, r_med = [], []

for i in range(len(z_bins)-1):
    mask = (zs >= z_bins[i]) & (zs < z_bins[i+1])
    if mask.sum() > 5:
        z_mid.append((z_bins[i] + z_bins[i+1]) / 2)
        r_med.append(np.median(rs[mask]))

ax.plot(z_mid, r_med, 'o-', lw=2, ms=7, label='Median size per redshift bin')

ax.set_xlabel('Spectroscopic redshift z', fontsize=12)
ax.set_ylabel('H-band effective radius Reff (kpc)', fontsize=12)
ax.set_title(
    'Galaxy Sizes Across Cosmic Time\n'
    '706 KMOS3D galaxies with measured H-band effective radii',
    fontsize=12
)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('hs_b_07_galaxy_sizes.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Redshift range: {zs.min():.2f} to {zs.max():.2f}')
print(f'Median effective radius: {np.median(rs):.2f} kpc')
print()
print('This plot describes the measured KMOS3D size sample within IntZ.')
print('The full IntZ corpus contains 1,292 records; 706 enter this size analysis.')
